# Optimización de Hiperparámetros — Dry Bean Classification
**Proyecto Reto: Clasificación Multiclase — Grupo 5**

Este notebook optimiza los hiperparámetros de los modelos ensemble utilizando `RandomizedSearchCV`:

1. Random Forest — búsqueda aleatoria
2. Gradient Boosting — búsqueda aleatoria
3. Comparación antes/después de la optimización
4. Guardado del modelo optimizado final

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from scipy.stats import randint, uniform

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
plt.style.use('dark_background')

## 1. Cargar datos y preparar splits

In [ ]:
df = pd.read_csv('../data/processed/dry_bean_clean.csv')
print('Shape:', df.shape)

X = df.drop('Class', axis=1)
y_raw = df['Class']

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Baseline — modelo sin optimizar

In [ ]:
rf_base = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
base_acc = accuracy_score(y_test, rf_base.predict(X_test))
print(f'Random Forest BASELINE — Test Accuracy: {base_acc:.4f}')

## 3. RandomizedSearchCV — Random Forest

In [ ]:
rf_param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', 0.5, 0.7, 0.9],
    'bootstrap': [True, False],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_search.fit(X_train, y_train)

print(f'\nMejores hiperparámetros RF:')
for k, v in rf_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor CV Score: {rf_search.best_score_:.4f}')

In [ ]:
rf_best = rf_search.best_estimator_
rf_train_acc = accuracy_score(y_train, rf_best.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_best.predict(X_test))
rf_diff = abs(rf_train_acc - rf_test_acc) * 100

print(f'Random Forest OPTIMIZADO:')
print(f'  Train: {rf_train_acc:.4f}  |  Test: {rf_test_acc:.4f}  |  Diff: {rf_diff:.2f}%')
print(f'  Overfitting: {"NO" if rf_diff < 5 else "SÍ (>5%)"}')

## 4. RandomizedSearchCV — Gradient Boosting

In [ ]:
gb_param_dist = {
    'n_estimators': randint(100, 500),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 10),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'subsample': uniform(0.7, 0.3),
    'max_features': ['sqrt', 'log2', 0.5, 0.7, 0.9],
}

gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_distributions=gb_param_dist,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

gb_search.fit(X_train, y_train)

print(f'\nMejores hiperparámetros GB:')
for k, v in gb_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor CV Score: {gb_search.best_score_:.4f}')

In [ ]:
gb_best = gb_search.best_estimator_
gb_train_acc = accuracy_score(y_train, gb_best.predict(X_train))
gb_test_acc = accuracy_score(y_test, gb_best.predict(X_test))
gb_diff = abs(gb_train_acc - gb_test_acc) * 100

print(f'Gradient Boosting OPTIMIZADO:')
print(f'  Train: {gb_train_acc:.4f}  |  Test: {gb_test_acc:.4f}  |  Diff: {gb_diff:.2f}%')
print(f'  Overfitting: {"NO" if gb_diff < 5 else "SÍ (>5%)"}')

## 5. Comparación antes/después

In [ ]:
comparison = pd.DataFrame({
    'Modelo': [
        'RF Baseline', 'RF Optimizado',
        'GB Optimizado',
    ],
    'Accuracy Train': [
        accuracy_score(y_train, rf_base.predict(X_train)),
        rf_train_acc,
        gb_train_acc,
    ],
    'Accuracy Test': [
        base_acc,
        rf_test_acc,
        gb_test_acc,
    ],
    'Diferencia (%)': [
        abs(accuracy_score(y_train, rf_base.predict(X_train)) - base_acc) * 100,
        rf_diff,
        gb_diff,
    ],
})

comparison = comparison.sort_values('Accuracy Test', ascending=False).reset_index(drop=True)
comparison.style.highlight_max(
    subset=['Accuracy Train', 'Accuracy Test'], color='#22c55e'
).highlight_min(
    subset=['Diferencia (%)'], color='#22c55e'
).format(precision=4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(comparison))
width = 0.35

ax.bar(x_pos - width/2, comparison['Accuracy Train'], width, label='Train', color='#4ade80')
ax.bar(x_pos + width/2, comparison['Accuracy Test'], width, label='Test', color='#60a5fa')

ax.set_ylabel('Accuracy')
ax.set_title('Optimización de Hiperparámetros — Antes vs Después', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(comparison['Modelo'])
ax.legend()
ax.set_ylim(0.85, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/optimization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Mejor modelo optimizado — Métricas completas

In [ ]:
best_name = comparison.iloc[0]['Modelo']
if 'RF' in best_name:
    best_model = rf_best
else:
    best_model = gb_best

y_pred = best_model.predict(X_test)

print(f'Mejor modelo: {best_name}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title(f'Matriz de Confusión — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/confusion_matrix_optimized.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.DataFrame({
        'Feature': X.columns,
        'Importancia': best_model.feature_importances_
    }).sort_values('Importancia', ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feat_imp['Feature'], feat_imp['Importancia'], color='#4ade80')
    ax.set_xlabel('Importancia')
    ax.set_title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/feature_importance_optimized.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Guardar modelo optimizado final

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

print(f'Modelo optimizado ({best_name}) guardado en ../models/best_model.pkl')
print('Scaler y LabelEncoder actualizados.')